# Exploratory Data Analysis

**CPU-only. Set `SAMPLE_SIZE` to limit rows for fast iteration.**

Target runtime: < 5 min on CPU with default sample size.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # no display needed
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────
SAMPLE_SIZE = 1000   # set -1 for full data
DATA_DIR = Path('../data/raw')
PLOT_DIR = Path('../results/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / 'train.csv'
TEST_FILE  = DATA_DIR / 'test.csv'
TARGET_COL = 'target'  # change to actual target column name

print(f'SAMPLE_SIZE = {SAMPLE_SIZE}')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
if TRAIN_FILE.exists():
    train = pd.read_csv(TRAIN_FILE, nrows=SAMPLE_SIZE if SAMPLE_SIZE > 0 else None)
    test  = pd.read_csv(TEST_FILE,  nrows=SAMPLE_SIZE if SAMPLE_SIZE > 0 else None)
else:
    # Synthetic fallback so notebook runs without real data
    print('WARNING: No data files found. Generating synthetic data for template demonstration.')
    rng = np.random.default_rng(42)
    n = SAMPLE_SIZE if SAMPLE_SIZE > 0 else 1000
    train = pd.DataFrame({
        'id': range(n),
        'feat_num_1': rng.normal(0, 1, n),
        'feat_num_2': rng.uniform(0, 100, n),
        'feat_cat_1': rng.choice(['A', 'B', 'C'], n),
        TARGET_COL: rng.integers(0, 2, n),
    })
    test = train.drop(columns=[TARGET_COL]).copy()

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head(3)

In [ ]:
# ── Basic stats ───────────────────────────────────────────────────
print('=== Train dtypes ===')
print(train.dtypes)
print()
print('=== Describe (numeric) ===')
train.describe()

In [ ]:
# ── Missing values ────────────────────────────────────────────────
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df.missing_count > 0].sort_values('missing_pct', ascending=False)

print(f'Columns with missing values: {len(missing_df)}')
if len(missing_df):
    print(missing_df)

In [ ]:
# ── Target distribution ───────────────────────────────────────────
if TARGET_COL in train.columns:
    print('=== Target Distribution ===')
    print(train[TARGET_COL].value_counts(normalize=True).round(4))
    
    fig, ax = plt.subplots(figsize=(6, 3))
    train[TARGET_COL].value_counts().plot(kind='bar', ax=ax)
    ax.set_title('Target Distribution')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'target_distribution.png', dpi=72)
    plt.close()
    print('Saved: results/plots/target_distribution.png')

In [ ]:
# ── Numeric feature distributions ─────────────────────────────────
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['id', TARGET_COL]]

print(f'Numeric features: {num_cols}')

if num_cols:
    n_cols = min(3, len(num_cols))
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 3))
    if n_cols == 1:
        axes = [axes]
    for ax, col in zip(axes, num_cols[:n_cols]):
        train[col].hist(bins=30, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'numeric_distributions.png', dpi=72)
    plt.close()
    print('Saved: results/plots/numeric_distributions.png')

In [ ]:
# ── Categorical features ──────────────────────────────────────────
cat_cols = train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical features: {cat_cols}')

for col in cat_cols[:5]:  # limit output
    n_unique = train[col].nunique()
    print(f'  {col}: {n_unique} unique values')
    if n_unique <= 20:
        print(train[col].value_counts().head(10))
    print()

In [ ]:
# ── Correlation matrix ────────────────────────────────────────────
if len(num_cols) >= 2:
    corr = train[num_cols].corr()
    
    fig, ax = plt.subplots(figsize=(max(4, len(num_cols)), max(3, len(num_cols)-1)))
    im = ax.imshow(corr.values, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticklabels(corr.columns)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'correlation_matrix.png', dpi=72)
    plt.close()
    print('Saved: results/plots/correlation_matrix.png')

In [ ]:
# ── Train/Test distribution comparison ───────────────────────────
common_num_cols = [c for c in num_cols if c in test.columns]

for col in common_num_cols[:3]:
    fig, ax = plt.subplots(figsize=(5, 2.5))
    train[col].hist(bins=20, ax=ax, alpha=0.6, label='train', density=True)
    test[col].hist(bins=20, ax=ax, alpha=0.6, label='test', density=True)
    ax.legend()
    ax.set_title(f'{col}: train vs test')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'train_test_{col}.png', dpi=72)
    plt.close()

print('Train/test comparison plots saved.')

## EDA Summary

Fill this after running EDA. Paste into `competition/dataset_info.md` and `research/findings.md`.

**Key observations:**
- TODO

**Missing value strategy:**
- TODO

**Suspicious features (potential leakage):**
- TODO

**Train/test distribution issues:**
- TODO